In [1]:
import pandas as pd
import re
from pathlib import Path

## Cleaning of Nigeria Property Centre Dataset

In [2]:
BASE_DIR = Path.cwd().parent  
RAW = BASE_DIR / "Data" / "raw"
df_original = pd.read_csv(RAW / "npc_abuja_rentals.csv")

In [3]:
df_original.head()

,Property Category,Price (Per Annum),Location,Scraped_At
0,Conference / meeting / training room for rent,"₦200,000","Kumasi Crescent, Wuse 2, Abuja",2026-06-01 11:07:00
1,2 bedroom flat / apartment for rent,"₦6,000,000","Jabi, Abuja",2026-06-01 11:07:00
2,3 bedroom detached bungalow for rent,"₦12,000,000","Lifecamp Main By Kado Round About, Life Camp, ...",2026-06-01 11:07:00
3,3 bedroom flat / apartment for rent,"₦6,000,000","Idu Industrial, Abuja",2026-06-01 11:07:00
4,Office space for rent,"₦180,000,000","Wuse 2, Abuja",2026-06-01 11:07:00


In [4]:
df = df_original.copy()
df.head()

,Property Category,Price (Per Annum),Location,Scraped_At
0,Conference / meeting / training room for rent,"₦200,000","Kumasi Crescent, Wuse 2, Abuja",2026-06-01 11:07:00
1,2 bedroom flat / apartment for rent,"₦6,000,000","Jabi, Abuja",2026-06-01 11:07:00
2,3 bedroom detached bungalow for rent,"₦12,000,000","Lifecamp Main By Kado Round About, Life Camp, ...",2026-06-01 11:07:00
3,3 bedroom flat / apartment for rent,"₦6,000,000","Idu Industrial, Abuja",2026-06-01 11:07:00
4,Office space for rent,"₦180,000,000","Wuse 2, Abuja",2026-06-01 11:07:00


In [5]:
# print all rows where the price contains a dollar sign
for price in df["Price (Per Annum)"]:
    if "$" in str(price):
        print(price)

$150,000
$45,000
$35,000
$50,000
$35,000
$30,000
$25,000
$35,000
$30,000
$35,000
$50,000
$35,000
$60,000
$35,000,000
$200
$55,000
$120,000
$90,000
$35,000
$65,000
$37,000
$16,000
$460,000
$70,000
$36,000
$60,000
$45,000
$57,000
$40,000
$40,000
$1,300,000
$60,000
$35,000
$35,000
$40,000
$65,000
$35,000
$35,000
$60,000
$35,000
$30,000


In [6]:
#convert all dollar prices to naira
exchange_rate = 1370.43 #exchange rate on 4th May, 2026

col = 'Price (Per Annum)'
df[col] = df[col].str.strip()
df[col] = df[col].str.replace(',', '')

#flag the rows that are in dollars
is_dollar = df[col].str.startswith('$')

#strip out the currency symbols
df[col] = df[col].str.replace('$', '', regex=False)
df[col] = df[col].str.replace('₦', '', regex=False)

#convert Price (Per Annum) to float
df[col] = df[col].astype(float)

df.loc[is_dollar, col] = df.loc[is_dollar, col] * exchange_rate
df.head()

,Property Category,Price (Per Annum),Location,Scraped_At
0,Conference / meeting / training room for rent,200000.0,"Kumasi Crescent, Wuse 2, Abuja",2026-06-01 11:07:00
1,2 bedroom flat / apartment for rent,6000000.0,"Jabi, Abuja",2026-06-01 11:07:00
2,3 bedroom detached bungalow for rent,12000000.0,"Lifecamp Main By Kado Round About, Life Camp, ...",2026-06-01 11:07:00
3,3 bedroom flat / apartment for rent,6000000.0,"Idu Industrial, Abuja",2026-06-01 11:07:00
4,Office space for rent,180000000.0,"Wuse 2, Abuja",2026-06-01 11:07:00


In [7]:
#create bedroom column
df['Bedrooms'] = df['Property Category'].str.extract(r'(\d+)')
df['Bedrooms'] = df['Bedrooms'].astype('Int64')
self_contain_mask = df['Property Category'].str.contains('Self contain', case=False, na=False)
df.loc[self_contain_mask, 'Bedrooms'] = 1
df[['Property Category', 'Bedrooms']].head(10)

,Property Category,Bedrooms
0,Conference / meeting / training room for rent,<NA>
1,2 bedroom flat / apartment for rent,2
2,3 bedroom detached bungalow for rent,3
3,3 bedroom flat / apartment for rent,3
4,Office space for rent,<NA>
5,5 bedroom terraced duplex for rent,5
6,4 bedroom terraced duplex for rent,4
7,3 bedroom terraced duplex for rent,3
8,2 bedroom flat / apartment for rent,2
9,1 bedroom flat / apartment for rent,1


In [8]:
# Remove the number and the word "bedroom" or "bedrooms" (handling spaces gracefully)
df['Property Category'] = df['Property Category'].str.replace(r'\d+\s*bedrooms?', '', regex=True, case=False)

# Remove the "for rent" text
df['Property Category'] = df['Property Category'].str.replace('for rent', '', case=False)
df['Property Category'] = df['Property Category'].str.strip()
df['Property Category'].unique()

array(['Conference / meeting / training room', 'flat / apartment',
       'detached bungalow', 'Office space', 'terraced duplex',
       'Commercial property', 'mini flat (room and parlour)',
       'detached duplex', 'semi-detached duplex',
       'Self contain (single rooms)', 'house', 'Shop',
       'semi-detached bungalow', 'commercial property',
       'Plaza / complex / mall', 'Warehouse', 'Commercial land',
       'hotel / guest house', 'Land', 'Event centre / venue',
       'Residential land', 'terraced bungalow', 'Mixed-use land'],
      dtype=object)

In [9]:
# List out non-residential keywords to remove
non_residential = "Conference / meeting / training room|Office space|Commercial property|Shop|commercial property|Plaza / complex / mall|Warehouse|Commercial land|hotel / guest house|Land|Event centre / venue|Residential land|Mixed-use land"

df = df[~df['Property Category'].str.contains(non_residential, case=False, na=False)]

print(df['Property Category'].value_counts())

Property Category
flat / apartment                769
terraced duplex                 427
detached duplex                 238
semi-detached duplex            125
Self contain (single rooms)      79
house                            78
mini flat (room and parlour)     71
detached bungalow                40
semi-detached bungalow           10
terraced bungalow                 1
Name: count, dtype: int64


In [10]:
Property_type_mapping = {
    'self contain (single rooms)': 'Self Contain',
    'mini flat (room and parlour)': 'Mini Flat',
    'flat / apartment': 'Apartment',
    'block of flats': 'Apartment',
    
    'detached duplex': 'Detached Duplex',
    'semi-detached duplex': 'Semi-Detached Duplex',
    'terraced duplex': 'Terraced Duplex',
    
    'detached bungalow': 'Detached Bungalow',
    'semi-detached bungalow': 'Semi-detached Bungalow',
    'terraced bungalow': 'Terraced Bungalow',
    
    'house': 'House' 
}

# Create new Property Type column
df['Property Type'] = df['Property Category'].str.lower().map(Property_type_mapping)

category_mapping = {
    'Self Contain': 'Apartment',
    'Mini Flat': 'Apartment',
    'Apartment': 'Apartment',
    
    'Detached Duplex': 'Duplex',
    'Semi-Detached Duplex': 'Duplex',
    'Terraced Duplex': 'Duplex',
    
    'Detached Bungalow': 'Bungalow',
    'Semi-detached Bungalow': 'Bungalow',
    'Terraced Bungalow': 'Bungalow',
    
    'House': 'House'
}

df['Property Category'] = df['Property Type'].map(category_mapping)

df[['Property Category', 'Property Type']].drop_duplicates().dropna()

,Property Category,Property Type
1,Apartment,Apartment
2,Bungalow,Detached Bungalow
5,Duplex,Terraced Duplex
11,Apartment,Mini Flat
17,Duplex,Detached Duplex
36,Duplex,Semi-Detached Duplex
41,Apartment,Self Contain
58,House,House
124,Bungalow,Semi-detached Bungalow
1752,Bungalow,Terraced Bungalow


In [11]:
#find null values
num_isnull = df.isnull().sum()
print(num_isnull)

Property Category    0
Price (Per Annum)    0
Location             0
Scraped_At           0
Bedrooms             0
Property Type        0
dtype: int64


In [ ]:
#create district column by extracting them from locaton column
import numpy as np
amac_districts = ['Jabi', 'Kaura', 'Garki', 'Kabusa', 'City Centre', 'Wuse', 'Gwarinpa', 'Gui', 'Karshi', 'Asokoro', 'Jahi', 'Guzape', 'Apo', 'Durumi', 'Lugbe', 'Lokogoma', 'Maitama', 'Wuye', 'Katampe', 'Life Camp', 'Utako', 'Mabushi', 'Idu', 'Kado', 'Galadimawa', 'Karu', 'Gaduwa', 'Gudu', 'kukwaba', 'Karmo', 'Kubwa', 'Central Business District', 'CBD']

district_pattern = '|'.join(amac_districts)

#extract the district name. 
df['District'] = df['Location'].str.extract(f'({district_pattern})', flags=re.IGNORECASE, expand=False)

df['District'] = df['District'].str.title()
print(df['District'].value_counts(dropna=False))

District
Maitama                      207
Katampe                      170
Guzape                       157
Jahi                         135
Gwarinpa                     122
Asokoro                      117
Life Camp                    111
Wuse                          92
Wuye                          85
Apo                           84
Jabi                          81
Mabushi                       71
Lugbe                         66
Kubwa                         36
Durumi                        34
Galadimawa                    34
Kado                          31
Garki                         31
Gaduwa                        30
Lokogoma                      30
Idu                           27
Kaura                         18
Utako                         15
Gudu                          14
NaN                           10
Kabusa                         8
Karu                           7
Karmo                          6
Kukwaba                        5
Central Business District      3
G

In [13]:
new_order = [
    'District', 
    'Bedrooms', 
    'Price (Per Annum)', 
    'Property Category', 
    'Property Type', 
    'Location',   
    'Scraped_At'    
]

df = df[new_order]

df.head()

,District,Bedrooms,Price (Per Annum),Property Category,Property Type,Location,Scraped_At
1,Jabi,2,6000000.0,Apartment,Apartment,"Jabi, Abuja",2026-06-01 11:07:00
2,Kado,3,12000000.0,Bungalow,Detached Bungalow,"Lifecamp Main By Kado Round About, Life Camp, ...",2026-06-01 11:07:00
3,Idu,3,6000000.0,Apartment,Apartment,"Idu Industrial, Abuja",2026-06-01 11:07:00
5,Jabi,5,15000000.0,Duplex,Terraced Duplex,"Citec, Jabi, Abuja",2026-06-01 11:07:00
6,Lugbe,4,5000000.0,Duplex,Terraced Duplex,"Von Axis, Lugbe District, Abuja",2026-06-01 11:07:00


In [14]:
num_isnull = df.isnull().sum()
print(num_isnull)

District             10
Bedrooms              0
Price (Per Annum)     0
Property Category     0
Property Type         0
Location              0
Scraped_At            0
dtype: int64


In [16]:
df.dropna(subset=['District'], inplace=True)
num_isnull = df.isnull().sum()
print(num_isnull)

District             0
Bedrooms             0
Price (Per Annum)    0
Property Category    0
Property Type        0
Location             0
Scraped_At           0
dtype: int64


In [17]:
df.head()

,District,Bedrooms,Price (Per Annum),Property Category,Property Type,Location,Scraped_At
1,Jabi,2,6000000.0,Apartment,Apartment,"Jabi, Abuja",2026-06-01 11:07:00
2,Kado,3,12000000.0,Bungalow,Detached Bungalow,"Lifecamp Main By Kado Round About, Life Camp, ...",2026-06-01 11:07:00
3,Idu,3,6000000.0,Apartment,Apartment,"Idu Industrial, Abuja",2026-06-01 11:07:00
5,Jabi,5,15000000.0,Duplex,Terraced Duplex,"Citec, Jabi, Abuja",2026-06-01 11:07:00
6,Lugbe,4,5000000.0,Duplex,Terraced Duplex,"Von Axis, Lugbe District, Abuja",2026-06-01 11:07:00


In [19]:
processed_dir = BASE_DIR / "Data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(BASE_DIR / "Data" / "processed" / "npc_cleaned.csv", index=False)